# Resumable Colab runner
Set the runtime to GPU. Completed `.jsonl.gz` fragments are persisted to Drive, so rerunning the final cell resumes at the next shard. Put `HF_TOKEN` in Colab Secrets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this after pushing the project to GitHub.
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'
PROJECT = '/content/video-indexer'
DRIVE_OUTPUT = '/content/drive/MyDrive/video-indexer/fragments'
DATASET_REPO = 'HuggingFaceFV/finevideo'
SHARD_START = 0
SHARD_COUNT = 50
VIDEO_COLUMN = 'video'
ID_COLUMN = 'id'
BATCH_SIZE = 16
SAMPLES_PER_SECOND = 0  # exact reference; change only after validation


In [ ]:
import os, pathlib, subprocess
if not pathlib.Path(PROJECT).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT, 'pull', '--ff-only'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt', 'huggingface_hub'], check=True)
subprocess.run(['python', f'{PROJECT}/prepare.py'], cwd=PROJECT, check=True)
pathlib.Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)


In [ ]:
# Confirm the GPU before starting a long run.
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0))


In [ ]:
import gzip, shutil, time
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
all_shards = sorted(name for name in api.list_repo_files(DATASET_REPO, repo_type='dataset') if name.endswith('.parquet'))
shards = all_shards[SHARD_START:SHARD_START + SHARD_COUNT]
print(f'Configured {len(shards)} shards')

for number, filename in enumerate(shards, 1):
    stem = pathlib.Path(filename).stem
    persisted = pathlib.Path(DRIVE_OUTPUT) / f'{stem}.jsonl.gz'
    if persisted.exists():
        print(f'[{number}/{len(shards)}] skip {filename}')
        continue
    print(f'[{number}/{len(shards)}] process {filename}')
    started = time.perf_counter()
    local_shard = pathlib.Path(hf_hub_download(DATASET_REPO, filename, repo_type='dataset', token=token, local_dir='/content/shards'))
    local_jsonl = pathlib.Path('/content') / f'{stem}.jsonl'
    command = [
        'python', f'{PROJECT}/run.py', '--input', str(local_shard), '--output', str(local_jsonl),
        '--device', 'cuda:0', '--batch-size', str(BATCH_SIZE),
        '--samples-per-second', str(SAMPLES_PER_SECOND),
        '--video-column', VIDEO_COLUMN, '--id-column', ID_COLUMN,
    ]
    subprocess.run(command, cwd=PROJECT, check=True)
    subprocess.run(['python', f'{PROJECT}/tools/validate_output.py', str(local_jsonl)], check=True)
    with local_jsonl.open('rb') as source, gzip.open(str(persisted) + '.tmp', 'wb', compresslevel=6) as target:
        shutil.copyfileobj(source, target)
    pathlib.Path(str(persisted) + '.tmp').replace(persisted)
    local_jsonl.unlink(missing_ok=True)
    pathlib.Path(str(local_jsonl) + '.completed').unlink(missing_ok=True)
    local_shard.unlink(missing_ok=True)
    print(f'done in {(time.perf_counter() - started) / 60:.1f} min')
